In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
统计 thalamus 整体中各类 marker 阳性细胞比例
用法:
    python thalamus_marker_ratio.py input.h5ad
输出:
    thalamus_marker_ratios.csv
"""
import sys
import scanpy as sc
import pandas as pd
import numpy as np



# ------------------ 读数据 ------------------
print("Loading data …")
adata = sc.read_h5ad('/data/work/05.cluster/FuseMap/20250106/thalamus_latent_embeddings_all_spatial_pretrain/dmt_leiden_20250108_1.h5ad')





In [3]:
names = {'gw13':[
 '14_A03591A1C3_WT202403310045.h5ad',
 '16_A03592A4C6_WT202403310044.h5ad',
 '18_B03602C4D6_WT202405020031.h5ad',
 '20_B03606F3G5_WT202405020032.h5ad',
 '22_B03606C4E6_WT202403310050.h5ad',
 '23_B03609A4D6_WT202404150263.h5ad',
 '27_B03610C1E3_WT202403310051.h5ad',
 '31_B03619A1D3_WT202403310052.h5ad',
 '35_B03619E4G6_WT202403310053.h5ad',
 '39_A03589A1D4_WT202403310046.h5ad',
 '43_A03590E1G4_WT202403310064.h5ad',
 '47_A03593C1F3_WT202403310068.h5ad',
 '51_B03605C2E5_WT202406020126.h5ad',
 '55_B03613E3G6_WT202403310069.h5ad',
 '59_B03612E4G6_WT202403310059.h5ad',
 '63_B03606C1E3_WT202403310061.h5ad',
 '67_A03595A1D3_WT202403310062.h5ad',
 '71_A03595A4D6_WT202403310063.h5ad',
 '75_D03468D1E3_WT202403310066.h5ad',
 '80_D03473D4E6_WT202403310070.h5ad',
 '84_B03423D1E3_WT202403310065.h5ad',
 '89_D03466A1C3_WT202403310058.h5ad',
 '99_D03470D1E3_WT202404290071.h5ad',
 '104_B03615F3G5_WT202405020035.h5ad',
 '105_D03468A4C6_WT202403310067.h5ad',],
    'gw13_rep': [ 'A03988A1C2_WT202407161208.h5ad',
 # 'A03591D4E5_WT2024071215074.h5ad',
 'A03590A3D6_WT202407192652.h5ad',
 'A03994F1G2_WT2024071215067.h5ad'],
    'gw10': ['A03587A5C6_WT2024071215080.h5ad'], # GW10
 'gw12': ['B03607C4E6_WT2024071214941.h5ad'], # GW12
    'gw16': ['B03618D3F6_WT202407152793.h5ad'], # GW16
        }

In [4]:
# 把基因名统一成大写，方便匹配
gene_names = [g.upper() for g in adata.var_names]

# 一个小函数：判断某基因是否表达（>0）
# -------------- 稀疏安全版：不 toarray --------------
from scipy.sparse import issparse

def positive(adata, gene):
    """返回该基因表达>0的布尔数组（稀疏矩阵也适用）"""
    try:
        idx = gene_names.index(gene.upper())
        vec = adata.X[:, idx]          # 这一步仍是稀疏或 dense 视图
        if issparse(vec):
            # 稀疏矩阵直接转 bool 再按行求和
            return np.array((vec > 0).todense()).ravel()
        else:
            return vec > 0
    except ValueError:
        raise ValueError(f"Gene {gene} not found!")

In [7]:
# ------------------ 定义要统计的组合 ------------------
combinations = {
    "SST+GAD1+"   : ["SST", "GAD1"],
    "SST+DLX2+"   : ["SST", "DLX2"],
    "SST+SOX14+"  : ["SST", "SOX14"],
    "SST+"        : ["SST"],
    "HES1+"       : ["HES1"],
    "NOTCH1+"     : ["NOTCH1"],
    "SOX2+"       : ["SOX2"],
    "LHX1+"       : ["LHX1"],
    "PAX6+"       : ["PAX6"],
    "KITLG+"      : ["KITLG"],
    "NTNG1+"      : ["NTNG1"],
    "TCF7L2+"     : ["TCF7L2"],
    "HES1+SOX2+PAX6+" : ["HES1", "SOX2", "PAX6"],
    'SST+DLX2+GAD1+': ['SST', 'DLX2', 'GAD1'],   
    'SST+SOX14+GAD1+': ['SST', 'SOX14', 'GAD1'],
    'PVALB': ['PVALB'], 
    'SST': ['SST'], 
    'VIP': ['VIP'], 
    # 'CCKM': ['CCKM'], 
    'ISL1': ['ISL1'], 
    'DLX2': ['DLX2'], 
    'SOX14': ['SOX14'], 
    'GAD1': ['GAD1'],
}

In [8]:
gw = 'gw13'
gws = adata[adata.obs['slice_code'].isin(names[gw])].copy()
n_total = gws.n_obs
res = {}
mask = None
for name, genes in combinations.items():
    mask = positive(gws, genes[0])
    for g in genes[1:]:
        mask &= positive(gws, g)
        # print(len(mask))
    res[name] = mask.sum() / n_total
df = pd.Series(res, name="proportion").reset_index()
df.rename(columns={"index": "combination"}, inplace=True)
# print(df)
df.to_csv(f"/data/work/05.cluster/FuseMap/20250117/08_percentage/{gw}.csv", index=False)

/opt/software/python/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [9]:
gw = 'gw13_rep'
gws = adata[adata.obs['slice_code'].isin(names[gw])].copy()
n_total = gws.n_obs
res = {}
mask = None
for name, genes in combinations.items():
    mask = positive(gws, genes[0])
    for g in genes[1:]:
        mask &= positive(gws, g)
    res[name] = mask.sum() / n_total
df = pd.Series(res, name="proportion").reset_index()
df.rename(columns={"index": "combination"}, inplace=True)
# print(df)
df.to_csv(f"/data/work/05.cluster/FuseMap/20250117/08_percentage/{gw}.csv", index=False)

In [10]:
gw = 'gw10'
gws = adata[adata.obs['slice_code'].isin(names[gw])].copy()
n_total = gws.n_obs
res = {}
mask = None
for name, genes in combinations.items():
    mask = positive(gws,genes[0])
    for g in genes[1:]:
        mask &= positive(gws, g)
    res[name] = mask.sum() / n_total
df = pd.Series(res, name="proportion").reset_index()
df.rename(columns={"index": "combination"}, inplace=True)
# print(df)
df.to_csv(f"/data/work/05.cluster/FuseMap/20250117/08_percentage/{gw}.csv", index=False)

In [11]:
gw = 'gw12'
gws = adata[adata.obs['slice_code'].isin(names[gw])].copy()
n_total = gws.n_obs
res = {}
mask = None
for name, genes in combinations.items():
    mask = positive(gws,genes[0])
    for g in genes[1:]:
        mask &= positive(gws, g)
    res[name] = mask.sum() / n_total
df = pd.Series(res, name="proportion").reset_index()
df.rename(columns={"index": "combination"}, inplace=True)
# print(df)
df.to_csv(f"/data/work/05.cluster/FuseMap/20250117/08_percentage/{gw}.csv", index=False)

In [12]:
gw = 'gw16'
gws = adata[adata.obs['slice_code'].isin(names[gw])].copy()
n_total = gws.n_obs
res = {}
mask = None
for name, genes in combinations.items():
    mask = positive(gws, genes[0])
    for g in genes[1:]:
        mask &= positive(gws, g)
    res[name] = mask.sum() / n_total
df = pd.Series(res, name="proportion").reset_index()
df.rename(columns={"index": "combination"}, inplace=True)
# print(df)
df.to_csv(f"/data/work/05.cluster/FuseMap/20250117/08_percentage/{gw}.csv", index=False)